# Day 7 - Market Basket Analysis

## Objective
Identify products frequently purchased together using
Association Rule Mining.

## Business Requirements
- Minimum Support: 1%
- Minimum Confidence: 30%
- Minimum Lift: Greater than 1.2

## Data Preparation
- Remove exact duplicate transaction rows.
- Exclude cancelled invoices.
- Exclude invalid transactions (Quantity <= 0 or UnitPrice <= 0).
- Retain invoices with missing CustomerID because customer identity
  is not required for invoice-level basket analysis.
- Treat each InvoiceNo as one shopping basket.

import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [1]:
import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
df = pd.read_excel("../data/raw/Online Retail.xlsx")

print("Original Shape:", df.shape)
display(df.head())

Original Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
# Create a separate copy
mba_df = df.drop_duplicates().copy()

# Convert InvoiceDate to datetime
mba_df["InvoiceDate"] = pd.to_datetime(mba_df["InvoiceDate"])

# Clean transaction identifiers
mba_df["InvoiceNo"] = mba_df["InvoiceNo"].astype(str).str.strip()
mba_df["StockCode"] = mba_df["StockCode"].astype(str).str.strip()

# Exclude cancelled invoices
mba_df = mba_df[
    ~mba_df["InvoiceNo"].str.startswith("C")
].copy()

# Keep valid purchase transactions
mba_df = mba_df[
    (mba_df["Quantity"] > 0) &
    (mba_df["UnitPrice"] > 0)
].copy()

# Remove missing or empty product codes
mba_df = mba_df[
    mba_df["StockCode"].notna() &
    (mba_df["StockCode"] != "")
].copy()

print("Rows After Cleaning:", len(mba_df))
print("Unique Invoices:", mba_df["InvoiceNo"].nunique())
print("Unique Products:", mba_df["StockCode"].nunique())
print("Unique Customers:", mba_df["CustomerID"].nunique())
print("Missing CustomerID:", mba_df["CustomerID"].isna().sum())

Rows After Cleaning: 524878
Unique Invoices: 19960
Unique Products: 3922
Unique Customers: 4338
Missing CustomerID: 132186


In [5]:
# Create one basket per invoice
transactions = (
    mba_df.groupby("InvoiceNo")["StockCode"]
    .apply(lambda products: sorted(set(products)))
)

# Convert Series to list of baskets
transactions_list = transactions.tolist()

print("Total Baskets:", len(transactions_list))

# Calculate basket statistics
basket_sizes = pd.Series(
    [len(basket) for basket in transactions_list]
)

print("\nBasket Statistics:")
print(basket_sizes.describe().round(2))

print("\nFirst 5 Baskets:")
for i, basket in enumerate(transactions_list[:5], start=1):
    print(f"Basket {i}: {basket}")

Total Baskets: 19960

Basket Statistics:
count    19960.00
mean        26.03
std         46.98
min          1.00
25%          6.00
50%         15.00
75%         29.00
max       1110.00
dtype: float64

First 5 Baskets:
Basket 1: ['21730', '22752', '71053', '84029E', '84029G', '84406B', '85123A']
Basket 2: ['22632', '22633']
Basket 3: ['21754', '21755', '21777', '22310', '22622', '22623', '22745', '22748', '22749', '48187', '84879', '84969']
Basket 4: ['22912', '22913', '22914', '22960']
Basket 5: ['21756']


In [6]:
print("Number of Baskets:", len(transactions_list))
print("Empty Baskets:", sum(len(basket) == 0 for basket in transactions_list))
print("Minimum Basket Size:", basket_sizes.min())
print("Maximum Basket Size:", basket_sizes.max())
print("Average Basket Size:", round(basket_sizes.mean(), 2))

# Check for duplicate products within baskets
duplicate_products = sum(
    len(basket) != len(set(basket))
    for basket in transactions_list
)

print("Baskets with Duplicate Products:", duplicate_products)

Number of Baskets: 19960
Empty Baskets: 0
Minimum Basket Size: 1
Maximum Basket Size: 1110
Average Basket Size: 26.03
Baskets with Duplicate Products: 0


In [7]:
# Count how many different invoices contain each product
product_audit = (
    mba_df.groupby("StockCode")
    .agg(
        Invoice_Count=("InvoiceNo", "nunique"),
        Description=("Description", "first")
    )
    .sort_values("Invoice_Count", ascending=False)
)

print("Top 30 Most Frequently Purchased Products:")
display(product_audit.head(30))

Top 30 Most Frequently Purchased Products:


,Invoice_Count,Description
StockCode,,
85123A,2198,WHITE HANGING HEART T-LIGHT HOLDER
85099B,2089,JUMBO BAG RED RETROSPOT
22423,1988,REGENCY CAKESTAND 3 TIER
47566,1685,PARTY BUNTING
20725,1565,LUNCH BAG RED RETROSPOT
84879,1455,ASSORTED COLOUR BIRD ORNAMENT
22197,1392,SMALL POPCORN HOLDER
22720,1385,SET OF 3 CAKE TINS PANTRY DESIGN
21212,1320,PACK OF 72 RETROSPOT CAKE CASES


In [8]:
# Identify the largest baskets
basket_size_df = transactions.str.len().rename(
    "Product_Count"
).reset_index()

largest_baskets = basket_size_df.sort_values(
    "Product_Count",
    ascending=False
).head(10)

display(largest_baskets)

,InvoiceNo,Product_Count
16355,573585,1110
19800,581219,749
19930,581492,731
19570,580729,721
9586,558475,704
19115,579777,687
19798,581217,676
522,537434,674
19571,580730,661
19372,580367,650


In [9]:
largest_invoice = largest_baskets.iloc[0]["InvoiceNo"]

print("Largest Invoice:", largest_invoice)

largest_invoice_items = (
    mba_df[mba_df["InvoiceNo"] == largest_invoice]
    [["StockCode", "Description", "Quantity", "UnitPrice"]]
    .drop_duplicates()
    .sort_values("StockCode")
)

display(largest_invoice_items)

Largest Invoice: 573585


,StockCode,Description,Quantity,UnitPrice
430237,11001,ASSTD DESIGN RACING CAR PEN,2,3.29
430238,15036,ASSORTED COLOURS SILK FAN,15,1.63
430239,15039,SANDALWOOD FAN,15,1.63
430240,15056BL,EDWARDIAN PARASOL BLACK,1,12.46
430241,15056N,EDWARDIAN PARASOL NATURAL,1,12.46
...,...,...,...,...
431345,90202B,WHITE ENAMEL FLOWER HAIR TIE,1,2.90
431346,90209C,PINK ENAMEL+GLASS HAIR COMB,3,2.07
431347,90214S,"LETTER ""S"" BLING KEY RING",1,0.83
431348,DOT,DOTCOM POSTAGE,1,2019.05


In [10]:
# Search for potentially non-merchandise transaction codes
keywords = (
    r"POSTAGE|DOTCOM|CARRIAGE|DISCOUNT|"
    r"BANK CHARGES|FEE|GIFT VOUCHER|"
    r"ADJUSTMENT|MANUAL|SAMPLES"
)

suspicious_products = (
    mba_df.groupby("StockCode")
    .agg(
        Description=("Description", "first"),
        Invoice_Count=("InvoiceNo", "nunique"),
        Total_Quantity=("Quantity", "sum"),
        Unit_Price=("UnitPrice", "median")
    )
    .reset_index()
)

suspicious_products = suspicious_products[
    suspicious_products["Description"]
    .str.contains(keywords, case=False, na=False)
].sort_values("Invoice_Count", ascending=False)

display(suspicious_products)

,StockCode,Description,Invoice_Count,Total_Quantity,Unit_Price
3914,POST,POSTAGE,1126,3150,18.000
3911,DOT,DOTCOM POSTAGE,706,706,185.470
2096,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,470,4800,4.950
442,21216,"SET 3 RETROSPOT TEA,COFFEE,SUGAR",430,2128,4.950
1834,22970,LONDON BUS COFFEE MUG,317,4334,2.550
3912,M,Manual,289,6984,1.950
1835,22971,QUEENS GUARD COFFEE MUG,272,2431,2.550
1202,22303,COFFEE MUG APPLES DESIGN,261,2816,2.550
1200,22301,COFFEE MUG CAT + BIRD DESIGN,252,2205,2.550
1201,22302,COFFEE MUG PEARS DESIGN,212,2329,2.550


In [11]:
non_standard_codes = (
    mba_df.groupby("StockCode")
    .agg(
        Description=("Description", "first"),
        Invoice_Count=("InvoiceNo", "nunique")
    )
    .reset_index()
)

non_standard_codes = non_standard_codes[
    ~non_standard_codes["StockCode"].str.match(
        r"^\d+[A-Za-z]?$",
        na=False
    )
].sort_values("Invoice_Count", ascending=False)

display(non_standard_codes)

,StockCode,Description,Invoice_Count
3914,POST,POSTAGE,1126
3911,DOT,DOTCOM POSTAGE,706
18,15056BL,EDWARDIAN PARASOL BLACK,321
3912,M,Manual,289
3903,C2,CARRIAGE,141
21,15056bl,EDWARDIAN PARASOL BLACK,62
3910,DCGSSGIRL,GIRLS PARTY BAG,13
3902,BANK CHARGES,Bank Charges,11
3909,DCGSSBOY,BOYS PARTY BAG,11
3917,gift_0001_20,Dotcomgiftshop Gift Voucher £20.00,9


In [12]:
# Normalize StockCode to uppercase
mba_df["StockCode"] = (
    mba_df["StockCode"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Define non-merchandise codes
excluded_codes = {
    "POST",
    "DOT",
    "M",
    "C2",
    "23444",
    "BANK CHARGES",
    "AMAZONFEE",
    "B",
    "S",
    "GIFT_0001_10",
    "GIFT_0001_20",
    "GIFT_0001_30",
    "GIFT_0001_40",
    "GIFT_0001_50",
    "22016"
}

# Measure excluded data before filtering
excluded_df = mba_df[
    mba_df["StockCode"].isin(excluded_codes)
].copy()

print("Rows to Exclude:", len(excluded_df))
print("Affected Invoices:", excluded_df["InvoiceNo"].nunique())
print("Excluded Products:", excluded_df["StockCode"].nunique())

# Keep merchandise products only
mba_product_df = mba_df[
    ~mba_df["StockCode"].isin(excluded_codes)
].copy()

print("\nRows After Product Filtering:", len(mba_product_df))
print("Remaining Invoices:", mba_product_df["InvoiceNo"].nunique())
print("Remaining Products:", mba_product_df["StockCode"].nunique())

Rows to Exclude: 2418
Affected Invoices: 2354
Excluded Products: 15

Rows After Product Filtering: 522460
Remaining Invoices: 19773
Remaining Products: 3797


In [13]:
transactions = (
    mba_product_df.groupby("InvoiceNo")["StockCode"]
    .apply(lambda products: sorted(set(products)))
)

transactions_list = transactions.tolist()

basket_sizes = pd.Series(
    [len(basket) for basket in transactions_list]
)

print("Total Baskets:", len(transactions_list))
print("Empty Baskets:", (basket_sizes == 0).sum())
print("Average Basket Size:", round(basket_sizes.mean(), 2))
print("Median Basket Size:", basket_sizes.median())
print("Maximum Basket Size:", basket_sizes.max())
print("Total Unique Products:", mba_product_df["StockCode"].nunique())

Total Baskets: 19773
Empty Baskets: 0
Average Basket Size: 26.16
Median Basket Size: 15.0
Maximum Basket Size: 1106
Total Unique Products: 3797


In [14]:
from mlxtend.preprocessing import TransactionEncoder

# Initialize encoder
te = TransactionEncoder()

# Convert transactions into a sparse matrix
te_array = te.fit(transactions_list).transform(
    transactions_list,
    sparse=True
)

# Convert sparse matrix into pandas DataFrame
basket_matrix = pd.DataFrame.sparse.from_spmatrix(
    te_array,
    columns=te.columns_
)

print("Basket Matrix Shape:", basket_matrix.shape)

print("\nData Types:")
print(basket_matrix.dtypes.value_counts())

print("\nFirst 5 Rows:")
display(basket_matrix.head())

Basket Matrix Shape: (19773, 3797)

Data Types:
Sparse[bool, False]    3797
Name: count, dtype: int64

First 5 Rows:


10002  10080  10120  10123C  10124A  10124G  10125  10133  10135  11001  \
0  False  False  False   False   False   False  False  False  False  False   
1  False  False  False   False   False   False  False  False  False  False   
2  False  False  False   False   False   False  False  False  False  False   
3  False  False  False   False   False   False  False  False  False  False   
4  False  False  False   False   False   False  False  False  False  False   

   15030  15034  15036  15039  15044A  15044B  15044C  15044D  15056BL  \
0  False  False  False  False   False   False   False   False    False   
1  False  False  False  False   False   False   False   False    False   
2  False  False  False  False   False   False   False   False    False   
3  False  False  False  False   False   False   False   False    False   
4  False  False  False  False   False   False   False   False    False   

   15056N  15056P  15058A  15058B  15058C  15060B  16008  16010  16011  16012  \
0   False   False   False   False   False   False  False  False  False  False   
1   False   False   False   False   False   False  False  False  False  False   
2   False   False   False   False   False   False  False  False  False  False   
3   False   False   False   False   False   False  False  False  False  False   
4   False   False   False   False   False   False  False  False  False  False   

   16014  16015  16016  16020C  16033  16043  16045  16046  16048  16049  \
0  False  False  False   False  False  False  False  False  False  False   
1  False  False  False   False  False  False  False  False  False  False   
2  False  False  False   False  False  False  False  False  False  False   
3  False  False  False   False  False  False  False  False  False  False   
4  False  False  False   False  False  False  False  False  False  False   

   16052  16054  16151A  16156L  16156S  16161G  16161M  16161P  16161U  \
0  False  False   False   False   False   False   False   False   False   
1  False  False   False   False   False   False   False   False   False   
2  False  False   False   False   False   False   False   False   False   
3  False  False   False   False   False   False   False   False   False   
4  False  False   False   False   False   False   False   False   False   

   16162L  16162M  16168M  16169E  16169K  16169M  16169N  16169P  16202A  \
0   False   False   False   False   False   False   False   False   False   
1   False   False   False   False   False   False   False   False   False   
2   False   False   False   False   False   False   False   False   False   
3   False   False   False   False   False   False   False   False   False   
4   False   False   False   False   False   False   False   False   False   

   16202B  16202E  16206B  16207A  16207B  16216  16218  16219  16225  16235  \
0   False   False   False   False   False  False  False  False  False  False   
1   False   False   False   False   False  False  False  False  False  False   
2   False   False   False   False   False  False  False  False  False  False   
3   False   False   False   False   False  False  False  False  False  False   
4   False   False   False   False   False  False  False  False  False  False   

   16236  16237  16238  16244B  16248B  16254  16258A  16259  17001  17003  \
0  False  False  False   False   False  False   False  False  False  False   
1  False  False  False   False   False  False   False  False  False  False   
2  False  False  False   False   False  False   False  False  False  False   
3  False  False  False   False   False  False   False  False  False  False   
4  False  False  False   False   False  False   False  False  False  False   

   17007B  17011F  17012A  17012B  17012C  17012D  17012E  17012F  17013D  \
0   False   False   False   False   False   False   False   False   False   
1   False   False   False   False   False   False   False   False   False   
2   False   False   False   False   False   False 

In [15]:
# Calculate matrix density
total_cells = basket_matrix.shape[0] * basket_matrix.shape[1]

non_zero_cells = basket_matrix.sparse.to_coo().nnz

density = (non_zero_cells / total_cells) * 100

print("Total Cells:", total_cells)
print("Non-Zero Cells:", non_zero_cells)
print("Matrix Density:", round(density, 4), "%")
print("Sparse Matrix:", basket_matrix.dtypes.iloc[0])

Total Cells: 75078081
Non-Zero Cells: 517210
Matrix Density: 0.6889 %
Sparse Matrix: Sparse[bool, False]


In [16]:
frequent_itemsets = apriori(
    basket_matrix,
    min_support=0.01,
    use_colnames=True,
    max_len=3,
    low_memory=True
)

# Add itemset length
frequent_itemsets["length"] = (
    frequent_itemsets["itemsets"].apply(len)
)

# Sort by support
frequent_itemsets = frequent_itemsets.sort_values(
    "support",
    ascending=False
).reset_index(drop=True)

print("Total Frequent Itemsets:", len(frequent_itemsets))

print("\nItemsets by Length:")
display(
    frequent_itemsets["length"]
    .value_counts()
    .sort_index()
)

print("\nTop 20 Frequent Itemsets:")
display(frequent_itemsets.head(20))


Total Frequent Itemsets: 2000

Itemsets by Length:


length
1    829
2    910
3    261
Name: count, dtype: int64


Top 20 Frequent Itemsets:


,support,itemsets,length
0,0.114550,frozenset({85123A}),1
1,0.105649,frozenset({85099B}),1
2,0.100541,frozenset({22423}),1
3,0.085217,frozenset({47566}),1
4,0.079148,frozenset({20725}),1
5,0.073585,frozenset({84879}),1
6,0.070399,frozenset({22197}),1
7,0.070045,frozenset({22720}),1
8,0.066758,frozenset({21212}),1
9,0.064937,frozenset({22383}),1


In [17]:
# Generate association rules
rules = association_rules(
    frequent_itemsets.drop(columns=["length"]),
    metric="confidence",
    min_threshold=0.30
)

print("Total Association Rules:", len(rules))

# Apply all BRD business requirements
filtered_rules = rules[
    (rules["support"] >= 0.01) &
    (rules["confidence"] >= 0.30) &
    (rules["lift"] > 1.2)
].copy()

# Sort by lift
filtered_rules = filtered_rules.sort_values(
    ["lift", "confidence"],
    ascending=False
).reset_index(drop=True)

print("\nRules After Applying BRD Thresholds:", len(filtered_rules))

display(
    filtered_rules[
        ["antecedents", "consequents", "support",
         "confidence", "lift"]
    ].head(20)
)

Total Association Rules: 1676

Rules After Applying BRD Thresholds: 1676


,antecedents,consequents,support,confidence,lift
0,frozenset({22916}),"frozenset({22917, 22918})",0.010267,0.856540,78.773800
1,"frozenset({22917, 22918})",frozenset({22916}),0.010267,0.944186,78.773800
2,"frozenset({22916, 22918})",frozenset({22917}),0.010267,0.953052,78.519542
3,frozenset({22917}),"frozenset({22916, 22918})",0.010267,0.845833,78.519542
4,"frozenset({22920, 22916})",frozenset({22917}),0.010115,0.952381,78.464286
5,frozenset({22917}),"frozenset({22920, 22916})",0.010115,0.833333,78.464286
6,"frozenset({22920, 22917})",frozenset({22916}),0.010115,0.934579,77.972318
7,frozenset({22916}),"frozenset({22920, 22917})",0.010115,0.843882,77.972318
8,frozenset({22916}),frozenset({22917}),0.011177,0.932489,76.825475
9,frozenset({22917}),frozenset({22916}),0.011177,0.920833,76.825475


In [18]:
product_names = (
    mba_product_df
    .groupby("StockCode")["Description"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    .to_dict()
)

def format_items(items):
    return " + ".join(
        f"{item}: {product_names.get(item, 'Unknown Product')}"
        for item in sorted(items)
    )

filtered_rules["Antecedent Products"] = (
    filtered_rules["antecedents"].apply(format_items)
)

filtered_rules["Consequent Products"] = (
    filtered_rules["consequents"].apply(format_items)
)

display(
    filtered_rules[
        ["Antecedent Products", "Consequent Products",
         "support", "confidence", "lift"]
    ].head(20)
)

,Antecedent Products,Consequent Products,support,confidence,lift
0,22916: HERB MARKER THYME,22917: HERB MARKER ROSEMARY + 22918: HERB MARK...,0.010267,0.856540,78.773800
1,22917: HERB MARKER ROSEMARY + 22918: HERB MARK...,22916: HERB MARKER THYME,0.010267,0.944186,78.773800
2,22916: HERB MARKER THYME + 22918: HERB MARKER ...,22917: HERB MARKER ROSEMARY,0.010267,0.953052,78.519542
3,22917: HERB MARKER ROSEMARY,22916: HERB MARKER THYME + 22918: HERB MARKER ...,0.010267,0.845833,78.519542
4,22916: HERB MARKER THYME + 22920: HERB MARKER ...,22917: HERB MARKER ROSEMARY,0.010115,0.952381,78.464286
5,22917: HERB MARKER ROSEMARY,22916: HERB MARKER THYME + 22920: HERB MARKER ...,0.010115,0.833333,78.464286
6,22917: HERB MARKER ROSEMARY + 22920: HERB MARK...,22916: HERB MARKER THYME,0.010115,0.934579,77.972318
7,22916: HERB MARKER THYME,22917: HERB MARKER ROSEMARY + 22920: HERB MARK...,0.010115,0.843882,77.972318
8,22916: HERB MARKER THYME,22917: HERB MARKER ROSEMARY,0.011177,0.932489,76.825475
9,22917: HERB MARKER ROSEMARY,22916: HERB MARKER THYME,0.011177,0.920833,76.825475


In [19]:
# Create a unique product bundle for each rule
rules_analysis = filtered_rules.copy()

rules_analysis["bundle_items"] = rules_analysis.apply(
    lambda row: row["antecedents"] | row["consequents"],
    axis=1
)

# Consolidate rules that represent the same product combination
bundle_summary = (
    rules_analysis
    .groupby("bundle_items")
    .agg(
        Support=("support", "max"),
        Max_Confidence=("confidence", "max"),
        Max_Lift=("lift", "max"),
        Rule_Variations=("bundle_items", "count")
    )
    .reset_index()
)

# Calculate approximate invoice count
bundle_summary["Invoice_Count"] = (
    bundle_summary["Support"] * len(transactions_list)
).round().astype(int)

# Add product descriptions
bundle_summary["Products"] = (
    bundle_summary["bundle_items"].apply(format_items)
)

bundle_summary["Number_of_Products"] = (
    bundle_summary["bundle_items"].apply(len)
)

# Sort by support to prioritize combinations with wider coverage
bundle_summary = bundle_summary.sort_values(
    ["Support", "Max_Lift"],
    ascending=False
).reset_index(drop=True)

display(
    bundle_summary[
        [
            "Products",
            "Number_of_Products",
            "Invoice_Count",
            "Support",
            "Max_Confidence",
            "Max_Lift",
            "Rule_Variations"
        ]
    ].head(20)
)

print("Unique Product Bundles:", len(bundle_summary))

,Products,Number_of_Products,Invoice_Count,Support,Max_Confidence,Max_Lift,Rule_Variations
0,22386: JUMBO BAG PINK POLKADOT + 85099B: JUMBO...,2,825,0.041724,0.677340,6.411222,2
1,22697: GREEN REGENCY TEACUP AND SAUCER + 22699...,2,767,0.038790,0.757157,14.057525,2
2,21931: JUMBO STORAGE BAG SUKI + 85099B: JUMBO ...,2,724,0.036616,0.611486,5.787900,2
3,22411: JUMBO SHOPPER VINTAGE RED PAISLEY + 850...,2,680,0.034390,0.578723,5.477787,2
4,20725: LUNCH BAG RED RETROSPOT + 22383: LUNCH ...,2,655,0.033126,0.510125,6.445172,2
5,20725: LUNCH BAG RED RETROSPOT + 20727: LUNCH ...,2,641,0.032418,0.503535,6.361915,2
6,22726: ALARM CLOCK BAKELIKE GREEN + 22727: ALA...,2,640,0.032367,0.653061,12.286374,2
7,22697: GREEN REGENCY TEACUP AND SAUCER + 22698...,2,632,0.031963,0.826144,16.125707,2
8,20725: LUNCH BAG RED RETROSPOT + 22384: LUNCH ...,2,606,0.030648,0.555963,7.024321,2
9,22698: PINK REGENCY TEACUP AND SAUCER + 22699:...,2,598,0.030243,0.781699,14.513184,2


Unique Product Bundles: 780


In [20]:
print("Total Association Rules:", len(rules))
print("Rules Meeting BRD Criteria:", len(filtered_rules))
print("Unique Product Bundles:", len(bundle_summary))

print(
    "Minimum Required Invoice Count:",
    int(np.ceil(0.01 * len(transactions_list)))
)

Total Association Rules: 1676
Rules Meeting BRD Criteria: 1676
Unique Product Bundles: 780
Minimum Required Invoice Count: 198


In [21]:
# Extract frequent 2-product combinations
pair_bundles = frequent_itemsets[
    frequent_itemsets["length"] == 2
].copy()

pair_bundles["Invoice_Count"] = (
    pair_bundles["support"] * len(transactions_list)
).round().astype(int)

pair_bundles["Products"] = (
    pair_bundles["itemsets"].apply(format_items)
)

pair_bundles = pair_bundles.sort_values(
    "support",
    ascending=False
).reset_index(drop=True)

print("Total 2-Product Combinations:", len(pair_bundles))

display(
    pair_bundles[
        ["Products", "Invoice_Count", "support"]
    ].head(20)
)

Total 2-Product Combinations: 910


,Products,Invoice_Count,support
0,22386: JUMBO BAG PINK POLKADOT + 85099B: JUMBO...,825,0.041724
1,22697: GREEN REGENCY TEACUP AND SAUCER + 22699...,767,0.038790
2,21931: JUMBO STORAGE BAG SUKI + 85099B: JUMBO ...,724,0.036616
3,22411: JUMBO SHOPPER VINTAGE RED PAISLEY + 850...,680,0.034390
4,20725: LUNCH BAG RED RETROSPOT + 22383: LUNCH ...,655,0.033126
5,20725: LUNCH BAG RED RETROSPOT + 20727: LUNCH ...,641,0.032418
6,22726: ALARM CLOCK BAKELIKE GREEN + 22727: ALA...,640,0.032367
7,22697: GREEN REGENCY TEACUP AND SAUCER + 22698...,632,0.031963
8,20725: LUNCH BAG RED RETROSPOT + 22384: LUNCH ...,606,0.030648
9,22698: PINK REGENCY TEACUP AND SAUCER + 22699:...,598,0.030243


In [22]:
# Keep only rules where one product predicts one other product
pair_rules = rules[
    (rules["antecedents"].apply(len) == 1) &
    (rules["consequents"].apply(len) == 1)
].copy()

# Create a lookup for directional confidence and lift
directional_metrics = {}

for _, row in pair_rules.iterrows():
    a = next(iter(row["antecedents"]))
    b = next(iter(row["consequents"]))

    directional_metrics[(a, b)] = {
        "Confidence": row["confidence"],
        "Lift": row["lift"]
    }

# Build a detailed table for frequent pairs
pair_results = []

for _, row in pair_bundles.iterrows():
    products = sorted(row["itemsets"])
    a, b = products

    ab = directional_metrics.get((a, b), {})
    ba = directional_metrics.get((b, a), {})

    pair_results.append({
        "Products": row["Products"],
        "Invoice_Count": row["Invoice_Count"],
        "Support": row["support"],
        "A_to_B_Confidence": ab.get("Confidence", np.nan),
        "B_to_A_Confidence": ba.get("Confidence", np.nan),
        "A_to_B_Lift": ab.get("Lift", np.nan),
        "B_to_A_Lift": ba.get("Lift", np.nan)
    })

pair_results_df = pd.DataFrame(pair_results)

# Apply BRD conditions to at least one direction
pair_results_df["Recommended"] = (
    (
        (pair_results_df["A_to_B_Confidence"] >= 0.30) &
        (pair_results_df["A_to_B_Lift"] > 1.2)
    ) |
    (
        (pair_results_df["B_to_A_Confidence"] >= 0.30) &
        (pair_results_df["B_to_A_Lift"] > 1.2)
    )
)

pair_results_df = pair_results_df.sort_values(
    ["Recommended", "Invoice_Count"],
    ascending=[False, False]
).reset_index(drop=True)

display(pair_results_df.head(20))

print(
    "Pairs Meeting Association Rule Conditions:",
    pair_results_df["Recommended"].sum()
)

,Products,Invoice_Count,Support,A_to_B_Confidence,B_to_A_Confidence,A_to_B_Lift,B_to_A_Lift,Recommended
0,22386: JUMBO BAG PINK POLKADOT + 85099B: JUMBO...,825,0.041724,0.677340,0.394926,6.411222,6.411222,True
1,22697: GREEN REGENCY TEACUP AND SAUCER + 22699...,767,0.038790,0.757157,0.720188,14.057525,14.057525,True
2,21931: JUMBO STORAGE BAG SUKI + 85099B: JUMBO ...,724,0.036616,0.611486,0.346577,5.787900,5.787900,True
3,22411: JUMBO SHOPPER VINTAGE RED PAISLEY + 850...,680,0.034390,0.578723,0.325515,5.477787,5.477787,True
4,20725: LUNCH BAG RED RETROSPOT + 22383: LUNCH ...,655,0.033126,0.418530,0.510125,6.445172,6.445172,True
5,20725: LUNCH BAG RED RETROSPOT + 20727: LUNCH ...,641,0.032418,0.409585,0.503535,6.361915,6.361915,True
6,22726: ALARM CLOCK BAKELIKE GREEN + 22727: ALA...,640,0.032367,0.653061,0.608944,12.286374,12.286374,True
7,22697: GREEN REGENCY TEACUP AND SAUCER + 22698...,632,0.031963,0.623889,0.826144,16.125707,16.125707,True
8,20725: LUNCH BAG RED RETROSPOT + 22384: LUNCH ...,606,0.030648,0.387220,0.555963,7.024321,7.024321,True
9,22698: PINK REGENCY TEACUP AND SAUCER + 22699:...,598,0.030243,0.781699,0.561502,14.513184,14.513184,True


Pairs Meeting Association Rule Conditions: 519


In [23]:
# Build final directional recommendations
recommendations = []

for _, row in pair_results_df.iterrows():

    candidates = []

    if (
        pd.notna(row["A_to_B_Confidence"]) and
        row["A_to_B_Confidence"] >= 0.30 and
        pd.notna(row["A_to_B_Lift"]) and
        row["A_to_B_Lift"] > 1.2
    ):
        candidates.append({
            "Direction": "A → B",
            "Confidence": row["A_to_B_Confidence"],
            "Lift": row["A_to_B_Lift"]
        })

    if (
        pd.notna(row["B_to_A_Confidence"]) and
        row["B_to_A_Confidence"] >= 0.30 and
        pd.notna(row["B_to_A_Lift"]) and
        row["B_to_A_Lift"] > 1.2
    ):
        candidates.append({
            "Direction": "B → A",
            "Confidence": row["B_to_A_Confidence"],
            "Lift": row["B_to_A_Lift"]
        })

    if candidates:
        best = max(candidates, key=lambda x: x["Confidence"])

        recommendations.append({
            "Products": row["Products"],
            "Invoice_Count": row["Invoice_Count"],
            "Support": row["Support"],
            "Best_Direction": best["Direction"],
            "Confidence": best["Confidence"],
            "Lift": best["Lift"]
        })

final_recommendations = pd.DataFrame(recommendations)

final_recommendations = final_recommendations.sort_values(
    ["Invoice_Count", "Confidence"],
    ascending=False
).reset_index(drop=True)

display(final_recommendations.head(20))

print("Total Qualified Pair Recommendations:", len(final_recommendations))

,Products,Invoice_Count,Support,Best_Direction,Confidence,Lift
0,22386: JUMBO BAG PINK POLKADOT + 85099B: JUMBO...,825,0.041724,A → B,0.677340,6.411222
1,22697: GREEN REGENCY TEACUP AND SAUCER + 22699...,767,0.038790,A → B,0.757157,14.057525
2,21931: JUMBO STORAGE BAG SUKI + 85099B: JUMBO ...,724,0.036616,A → B,0.611486,5.787900
3,22411: JUMBO SHOPPER VINTAGE RED PAISLEY + 850...,680,0.034390,A → B,0.578723,5.477787
4,20725: LUNCH BAG RED RETROSPOT + 22383: LUNCH ...,655,0.033126,B → A,0.510125,6.445172
5,20725: LUNCH BAG RED RETROSPOT + 20727: LUNCH ...,641,0.032418,B → A,0.503535,6.361915
6,22726: ALARM CLOCK BAKELIKE GREEN + 22727: ALA...,640,0.032367,A → B,0.653061,12.286374
7,22697: GREEN REGENCY TEACUP AND SAUCER + 22698...,632,0.031963,B → A,0.826144,16.125707
8,20725: LUNCH BAG RED RETROSPOT + 22384: LUNCH ...,606,0.030648,B → A,0.555963,7.024321
9,22698: PINK REGENCY TEACUP AND SAUCER + 22699:...,598,0.030243,A → B,0.781699,14.513184


Total Qualified Pair Recommendations: 519


In [24]:
# Final MBA validation
mba_validation = {
    "Total_Invoices": len(transactions_list),
    "Total_Products": basket_matrix.shape[1],
    "Total_Frequent_Itemsets": len(frequent_itemsets),
    "Single_Product_Itemsets": int(
        (frequent_itemsets["length"] == 1).sum()
    ),
    "Two_Product_Itemsets": int(
        (frequent_itemsets["length"] == 2).sum()
    ),
    "Three_Product_Itemsets": int(
        (frequent_itemsets["length"] == 3).sum()
    ),
    "Total_Association_Rules": len(rules),
    "Qualified_Rules": len(filtered_rules),
    "Unique_Product_Bundles": len(bundle_summary),
    "Qualified_Pair_Recommendations": len(final_recommendations)
}

display(
    pd.DataFrame(
        list(mba_validation.items()),
        columns=["Metric", "Value"]
    )
)

,Metric,Value
0,Total_Invoices,19773
1,Total_Products,3797
2,Total_Frequent_Itemsets,2000
3,Single_Product_Itemsets,829
4,Two_Product_Itemsets,910
5,Three_Product_Itemsets,261
6,Total_Association_Rules,1676
7,Qualified_Rules,1676
8,Unique_Product_Bundles,780
9,Qualified_Pair_Recommendations,519


In [25]:
from pathlib import Path

output_dir = Path("../outputs/mba")
output_dir.mkdir(parents=True, exist_ok=True)

filtered_rules.to_csv(
    output_dir / "association_rules.csv",
    index=False
)

bundle_summary.to_csv(
    output_dir / "product_bundles.csv",
    index=False
)

final_recommendations.to_csv(
    output_dir / "top_pair_recommendations.csv",
    index=False
)

print("MBA outputs saved successfully.")

MBA outputs saved successfully.
